# Uninformed search

### Maze Environments
The environments used is **SmallMaze** (visible in the figure).

<img src="images/maze.png" width="300">

The agent starts in cell $(0, 2)$ and has to reach the treasure in $(4, 3)$.

In order to use the environment we need first to import the packages of OpenAI Gym.

In [1]:
import os, sys, time, math

module_path = os.path.abspath(os.path.join('../tools'))
if module_path not in sys.path:
    sys.path.append(module_path)

from utils.ai_lab_functions import *
import gym, envs

### BFS Tree Search

In [2]:
def BFS_TreeSearch(problem):
    node = Node(problem.startstate, None)
    time_cost = 1
    space_cost = 1
    
    if node.state == problem.goalstate:
        return build_path(node), time_cost, space_cost
    
    frontier = NodeQueue()
    frontier.add(node)    
    
    while not frontier.is_empty():
        node = frontier.remove()
        
        for action in range(problem.action_space.n):
            
            child = Node(problem.sample(node.state, action), node)
            time_cost += 1
            
            if problem.goalstate == child.state: 
                return build_path(child), time_cost, space_cost
            
            frontier.add(child)
             
        space_cost = max(space_cost, len(frontier))  

    return None, time_cost, space_cost

### BFS Graph Search

In [3]:
def BFS_GraphSearch(problem):
    node = Node(problem.startstate, None)
    time_cost = 1
    space_cost = 1
    
    if node.state == problem.goalstate:
        return build_path(node), time_cost, space_cost
    
    frontier = NodeQueue()
    explored = set()
    frontier.add(node)
    
    while not frontier.is_empty():
        node = frontier.remove()
        explored.add(node.state)
        
        for action in range(problem.action_space.n):
            child = Node(problem.sample(node.state, action), node)
            time_cost += 1
            
            if not (child.state in explored or child.state in frontier):
                if problem.goalstate == child.state: 
                    return build_path(child), time_cost, space_cost
                frontier.add(child)
             
        space_cost = max(space_cost, len(frontier) + len(explored))
    
    return build_path(node), time_cost, space_cost     

### Check the results

In [4]:
envname = "SmallMaze-v0"
environment = gym.make(envname)

solution_ts, time_ts, memory_ts = BFS_TreeSearch(environment)
solution_gs, time_gs, memory_gs = BFS_GraphSearch(environment)

results = CheckResult_L1A1([solution_ts, time_ts, memory_ts], [solution_gs, time_gs, memory_gs], environment)
results.check_sol_ts()
results.check_sol_gs()

##########################################
#######  BFS TREE SEARCH PROBLEM  ########
##########################################
Solution: [(np.int64(0), np.int64(1)), (np.int64(0), np.int64(0)), (np.int64(1), np.int64(0)), (np.int64(2), np.int64(0)), (np.int64(3), np.int64(0)), (np.int64(4), np.int64(0)), (np.int64(4), np.int64(1)), (np.int64(4), np.int64(2)), (np.int64(4), np.int64(3))]
N° of nodes explored: 103723
Max n° of nodes in memory: 77791

##########################################
#######  BFS Graph SEARCH PROBLEM  #######
##########################################
Your solution: [(np.int64(0), np.int64(1)), (np.int64(0), np.int64(0)), (np.int64(1), np.int64(0)), (np.int64(2), np.int64(0)), (np.int64(3), np.int64(0)), (np.int64(4), np.int64(0)), (np.int64(4), np.int64(1)), (np.int64(4), np.int64(2)), (np.int64(4), np.int64(3))]
N° of nodes explored: 59
Max n° of nodes in memory: 15

===> Your solution is correct!



### Recursive DLS Tree Search

In [5]:
def Recursive_DLS_TreeSearch(node, problem, limit, explored=None):
    if problem.goalstate == node.state: # Goal state check
        return build_path(node), 1, node.depthcost
    elif limit == 0: # Limit budget check for cutoff
        return "cut_off", 1, node.depthcost
    
    space_cost = node.depthcost
    time_cost = 1 
    
    cut_off_occurred = False
    for action in range(problem.action_space.n): # Look around
        child = Node(problem.sample(node.state, action), node) # Child node 
        result = Recursive_DLS_TreeSearch(child, problem, limit-1) # Recursive call
        time_cost += result[1]

        space_cost = max(space_cost, result[2])
        
        if result[0] == "cut_off":
            cut_off_occurred = True
        elif result[0] != "failure": # Solution found
            return result[0], time_cost, space_cost
        
    if(cut_off_occurred): # Solution not found but cutoff occurred
        return "cut_off", time_cost, space_cost
    else: # No solution and no cutoff: failure
        return "failure", time_cost, space_cost

### Depth-Limited-Search

In [6]:
def DLS(problem, limit, RDLS_Function):
    node = Node(problem.startstate, None)
    return RDLS_Function(node, problem, limit, set())

### Iterative Deepening Search

In [7]:
def IDS(problem, DLS_Function):
    start = Node(problem.startstate)
    node = Node(problem.startstate, None)
    
    total_time_cost = 0
    total_space_cost = 1
    
    for i in zero_to_infinity():
        result = DLS(problem, i, DLS_Function)
        
        total_time_cost += result[1]
        total_space_cost = max(total_space_cost, result[2])
        
        if result[0] != "cut_off":
            return result[0], total_time_cost, total_space_cost, i

    return None, total_time_cost, total_space_cost, 0

### Check the result

In [8]:
envname = "SmallMaze-v0"
environment = gym.make(envname)

solution_ts, time_ts, memory_ts, iterations_ts = IDS(environment, Recursive_DLS_TreeSearch)

results = CheckResult_L1A2([solution_ts, time_ts, memory_ts, iterations_ts], environment)
results.check_sol_ts()

##########################################
#######  IDS TREE SEARCH PROBLEM  ########
##########################################
Necessary Iterations: 9
Your solution: [(np.int64(0), np.int64(1)), (np.int64(0), np.int64(0)), (np.int64(1), np.int64(0)), (np.int64(2), np.int64(0)), (np.int64(3), np.int64(0)), (np.int64(4), np.int64(0)), (np.int64(4), np.int64(1)), (np.int64(4), np.int64(2)), (np.int64(4), np.int64(3))]
N° of nodes explored: 138298
Max n° of nodes in memory: 9

===> Your solution is correct!

